# 03 — RAG Üretimi ve Yanıt Değerlendirme (İP-4, İP-6)

Kazanılan erişim sonuçlarını sabit LLM'e bağlar: her sistem için Top-k kaynak + yanıt üretir, yanıtları kaydeder.

**Değişmez kural:** Tüm sistemlerde aynı LLM, prompt (`prompts/rag_tr.txt`), Top-k ve üretim ayarları. Fark yalnızca erişim katmanından gelir (RQ3).

In [1]:
import os, sys
REPO_URL = 'https://github.com/EmreK37/RAG-Tabanli-Turkce-Soru-Cevap-Sistemleri.git'
BRANCH = 'RAG-dev'
REPO_DIR = '/content/RAG-proje'
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull origin {BRANCH}
!pip install -q -r requirements.txt
sys.path.insert(0, REPO_DIR)

from src.config import load_config
from src.data import jsonl_oku, jsonl_yaz, korpus_yukle, yonetmelik_parcala
from src.embeddings import DenseEmbedder
from src.retrieval import Bm25Retriever, FaissRetriever
from src.generation import RagGenerator

cfg = load_config()
chunklar = yonetmelik_parcala(
    korpus_yukle(cfg['veri']['korpus']),
    boyut=cfg['parcalama']['boyut_karakter'],
    ortusme=cfg['parcalama']['ortusme_karakter'],
)
chunk_metin = {c.chunk_id: c.metin for c in chunklar}
sorular = [s for s in jsonl_oku(cfg['veri']['sorular']) if s.get('bolme', 'test') == 'test']
# Her soru tipinden (olgusal, paraphrase, bicimbilimsel, cok-maddeli) dengeli 5'er soru = 20 test sorusu
ornek_sorular = []
for tip in ['olgusal', 'paraphrase', 'bicimbilimsel', 'cok-maddeli']:
    ornek_sorular.extend([s for s in sorular if s['tip'] == tip][:5])
print(f'Toplam test sorusu: {len(sorular)} | RAG üretim örneklemi (dengeli): {len(ornek_sorular)}')

Cloning into '/content/RAG-proje'...
remote: Enumerating objects: 117, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 117 (delta 44), reused 93 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (117/117), 106.62 KiB | 2.60 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/content/RAG-proje
From https://github.com/EmreK37/RAG-Tabanli-Turkce-Soru-Cevap-Sistemleri
 * branch            RAG-dev    -> FETCH_HEAD
Already up to date.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Toplam test sorusu: 120 | RAG üretim örneklemi (dengeli): 20


In [2]:
# Üreticiyi yükle (4-bit; ilk seferde model indirilir, ~5 GB)
uretici = RagGenerator(
    model_adi=cfg['uretim']['llm'],
    dort_bit=(cfg['uretim']['yukleme'] == '4bit'),
    maks_yeni_token=cfg['uretim']['maks_yeni_token'],
)
print('LLM hazır:', cfg['uretim']['llm'])

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

LLM hazır: Qwen/Qwen2.5-7B-Instruct


In [3]:
# RQ3 için 4 temsilci erişim sistemi (bm25, minilm, tr-model, e5-base) + sabit LLM ile yanıt üretimi
secili_modeller = ['bm25', 'minilm', 'tr-model', 'e5-base']
sistemler = {}
embedder_map = {}
for m in cfg['modeller']:
    ad = m['ad']
    if ad not in secili_modeller:
        continue
    if m['tur'] == 'sozcuksel':
        sistemler[ad] = Bm25Retriever(chunklar)
    else:
        emb = DenseEmbedder(
            m['hf'],
            sorgu_oneki=m.get('sorgu_oneki', ''),
            belge_oneki=m.get('belge_oneki', ''),
            sorgu_yonergesi=m.get('sorgu_yonergesi', ''),
        )
        embedder_map[ad] = emb
        sistemler[ad] = FaissRetriever(emb, chunklar)

top_k = cfg['erisim']['ana_top_k']
kayitlar = []
for ad, sistem in sistemler.items():
    print(f'=== Üretim başlıyor: {ad} ({len(ornek_sorular)} soru) ===')
    for soru in ornek_sorular:
        bulunan = sistem.retrieve(soru['soru'], k=top_k)
        kaynak_idler = [cid for cid, _ in bulunan]
        kaynaklar = [chunk_metin[cid] for cid in kaynak_idler]
        yanit = uretici.uret(soru['soru'], kaynaklar)
        kayitlar.append({
            'soru_id': soru['soru_id'],
            'tip': soru['tip'],
            'sistem': ad,
            'soru': soru['soru'],
            'kanit_chunk_id': soru['kanit_chunk_id'],
            'kaynaklar': kaynak_idler,
            'hit@5': 1.0 if soru['kanit_chunk_id'] in kaynak_idler else 0.0,
            'yanit': yanit,
            'referans': soru.get('referans_yanit', ''),
        })
    print(f'{ad} örnek yanıt ({ornek_sorular[0]["soru_id"]}):', kayitlar[-len(ornek_sorular)]['yanit'][:120], '...')

jsonl_yaz(kayitlar, 'results/yanitlar-rq3.jsonl')
print(f'Toplam {len(kayitlar)} yanıt kaydedildi: results/yanitlar-rq3.jsonl')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (34 outputs left out)

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

=== Üretim başlıyor: minilm (20 soru) ===
minilm örnek yanıt (S001): Doğuş Üniversitesi Önlisans ve Lisans Eğitim-Öğretim ve Sınav Yönetmeliği'nin temel amacını düzenlemektir, bu kapsam içi ...
=== Üretim başlıyor: e5-base (20 soru) ===
e5-base örnek yanıt (S001): Bu Yönetmelik, Doğuş Üniversitesi'nin önlisans ve lisans düzeyinde eğitim-öğretim, sınav ve değerlendirmelerine ilişkin  ...
=== Üretim başlıyor: tr-model (20 soru) ===
tr-model örnek yanıt (S001): Doğuş Üniversitesi Önlisans ve Lisans Eğitim-Öğretim ve Sınav Yönetmeliği'nin temel amacını, Doğuş Üniversitesi'nde hazı ...
=== Üretim başlıyor: bm25 (20 soru) ===
bm25 örnek yanıt (S001): Doğuş Üniversitesi Önlisans ve Lisans Eğitim-Öğretim ve Sınav Yönetmeliğinin temel amacını düzenlemektir, bu süreçte haz ...
Toplam 80 yanıt kaydedildi: results/yanitlar-rq3.jsonl


## Yanıt değerlendirme (faithfulness)

- Üretilen her yanıt için kaynaklar kaydedildiği için faithfulness, yanıtın iddialarının kaynaklarda desteklenip desteklenmediğine bakarak ölçülür.
- Pratik yol: `ragas` ile LLM-as-judge (faithfulness + answer correctness). **Judge modeli, üretim LLM'inden farklı ve sabit sürümlü olmalı**; hangi modelin judge olduğu rapora yazılır.
- RQ3 analizi: soru bazında retrieval skoru (notebook 02) ile yanıt doğruluğunu eşleştirip korelasyon bakılır; doğrusal olmayan ilişki beklenen bulgudur (H3).

In [4]:
# RQ3 Değerlendirmesi: Faithfulness (Kaynağa Bağlılık), Answer Correctness ve Atıf Doğruluğu
import re
import numpy as np
import pandas as pd

_TOK = re.compile(r'\w+', re.UNICODE)
_STOP = {'ve', 'ile', 'bir', 'bu', 'için', 'olarak', 'olan', 'göre', 'veya', 'da', 'de', 'ise', 'en', 'her', 'ne', 'kadar'}

def icerik_tokenlari(metin: str) -> set[str]:
    return {t for t in _TOK.findall(metin.lower()) if len(t) > 2 and t not in _STOP}

def kaynak_baglilik_skoru(yanit: str, kaynak_idler: list[str]) -> float:
    """Yanıttaki içerik terimlerinin getirilen kaynaklarda desteklenme oranı (faithfulness)."""
    if 'verilen kaynaklarda yer almıyor' in yanit.lower():
        return 1.0  # kaynakta yoksa doğru kaçınma (abstention) sadıktır
    y_tok = icerik_tokenlari(yanit)
    if not y_tok:
        return 0.0
    k_metin = ' '.join(chunk_metin[c] for c in kaynak_idler if c in chunk_metin)
    k_tok = icerik_tokenlari(k_metin)
    # Türkçe ek kök uyumu için ilk 5 karakter (gövde) eşleşmesi de kontrol edilir
    k_kokler = {t[:5] for t in k_tok if len(t) >= 5}
    desteklenen = sum(1 for t in y_tok if (t in k_tok) or (len(t) >= 5 and t[:5] in k_kokler))
    return desteklenen / len(y_tok)

def token_f1(yanit: str, referans: str) -> float:
    y_tok = icerik_tokenlari(yanit)
    r_tok = icerik_tokenlari(referans)
    if not y_tok or not r_tok:
        return 0.0
    y_kok = {t[:5] for t in y_tok if len(t) >= 5} | y_tok
    r_kok = {t[:5] for t in r_tok if len(t) >= 5} | r_tok
    ortak = len(y_kok & r_kok)
    if ortak == 0:
        return 0.0
    p = ortak / len(y_kok)
    r = ortak / len(r_kok)
    return 2 * p * r / (p + r)

# Anlamsal doğruluk için sabit değerlendirici (e5-base kosinüs benzerliği + sözcüksel F1 ortalaması)
judge_emb = embedder_map['e5-base']
yanit_vektorler = judge_emb.belgeleri_kodla([k['yanit'] for k in kayitlar])
ref_vektorler = judge_emb.belgeleri_kodla([k['referans'] for k in kayitlar])
anlamsal_skorlar = np.sum(yanit_vektorler * ref_vektorler, axis=1)

for k, sem_skor in zip(kayitlar, anlamsal_skorlar):
    k['faithfulness'] = round(float(kaynak_baglilik_skoru(k['yanit'], k['kaynaklar'])), 4)
    f1_skor = token_f1(k['yanit'], k['referans'])
    k['answer_correctness'] = round(float(0.6 * sem_skor + 0.4 * f1_skor), 4)
    hedef_madde = k['kanit_chunk_id'].split('-')[0] + '-' + k['kanit_chunk_id'].split('-')[1]
    k['dogru_atif'] = 1.0 if hedef_madde in k['yanit'].upper().replace(' ', '-') else 0.0

df_rq3 = pd.DataFrame(kayitlar)
rq3_ozet = df_rq3.groupby('sistem')[['hit@5', 'faithfulness', 'answer_correctness', 'dogru_atif']].mean().round(4)
rq3_ozet = rq3_ozet.sort_values('hit@5', ascending=False)
rq3_ozet.to_csv('results/rq3-rag-ozet.csv')
print('=== RQ3: Retrieval Başarısı vs Yanıt Kalitesi ve Faithfulness ===')
print(rq3_ozet)

print('\n=== Kanıt Bulunan (Hit@5=1) vs Bulunamayan (Hit@5=0) Sorularda Yanıt Kalitesi ===')
print(df_rq3.groupby('hit@5')[['faithfulness', 'answer_correctness', 'dogru_atif']].mean().round(4))

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

=== RQ3: Retrieval Başarısı vs Yanıt Kalitesi ve Faithfulness ===
          hit@5  faithfulness  answer_correctness  dogru_atif
sistem                                                       
e5-base    0.95        0.8964              0.6712        0.75
tr-model   0.70        0.8832              0.6662        0.45
bm25       0.65        0.8843              0.6630        0.60
minilm     0.65        0.8597              0.6382        0.60

=== Kanıt Bulunan (Hit@5=1) vs Bulunamayan (Hit@5=0) Sorularda Yanıt Kalitesi ===
       faithfulness  answer_correctness  dogru_atif
hit@5                                              
0.0          0.8684              0.5504      0.0952
1.0          0.8854              0.6985      0.7797


In [ ]:
# Eksik Modeller (bge-m3, qwen3-0.6b), Adaptif Router ve A4 Top-k (1, 3, 5) RAG Ablasyonu
from src.metrics import kacinma_isabeti

ek_modeller = ["bge-m3", "qwen3-0.6b"]
for m in cfg["modeller"]:
    ad = m["ad"]
    if ad in ek_modeller and ad not in sistemler:
        emb = DenseEmbedder(
            m["hf"],
            sorgu_oneki=m.get("sorgu_oneki", ""),
            belge_oneki=m.get("belge_oneki", ""),
            sorgu_yonergesi=m.get("sorgu_yonergesi", ""),
        )
        embedder_map[ad] = emb
        sistemler[ad] = FaissRetriever(emb, chunklar)

ek_kayitlar = []
for ad in ek_modeller:
    sistem = sistemler[ad]
    print(f"=== Ek RAG Üretimi: {ad} ({len(ornek_sorular)} soru) ===")
    for soru in ornek_sorular:
        bulunan = sistem.retrieve(soru["soru"], k=top_k)
        kaynak_idler = [cid for cid, _ in bulunan]
        kaynaklar = [chunk_metin[cid] for cid in kaynak_idler]
        yanit = uretici.uret(soru["soru"], kaynaklar)
        ek_kayitlar.append({
            "soru_id": soru["soru_id"],
            "tip": soru["tip"],
            "sistem": ad,
            "soru": soru["soru"],
            "kanit_chunk_id": soru["kanit_chunk_id"],
            "kaynaklar": kaynak_idler,
            "hit@5": 1.0 if soru["kanit_chunk_id"] in kaynak_idler else 0.0,
            "yanit": yanit,
            "referans": soru.get("referans_yanit", ""),
        })

# Adaptif Router (paraphrase/uzun sorgularda e5-base, diğerlerinde bm25) RAG yanıt kalitesi
for soru in ornek_sorular:
    secilen = "e5-base" if soru["tip"] == "paraphrase" or len(soru["soru"].split()) > 9 else "bm25"
    bulunan = sistemler[secilen].retrieve(soru["soru"], k=top_k)
    kaynak_idler = [cid for cid, _ in bulunan]
    kaynaklar = [chunk_metin[cid] for cid in kaynak_idler]
    yanit = uretici.uret(soru["soru"], kaynaklar)
    ek_kayitlar.append({
        "soru_id": soru["soru_id"],
        "tip": soru["tip"],
        "sistem": "router-rf",
        "soru": soru["soru"],
        "kanit_chunk_id": soru["kanit_chunk_id"],
        "kaynaklar": kaynak_idler,
        "hit@5": 1.0 if soru["kanit_chunk_id"] in kaynak_idler else 0.0,
        "yanit": yanit,
        "referans": soru.get("referans_yanit", ""),
    })

yanit_vek = judge_emb.belgeleri_kodla([k["yanit"] for k in ek_kayitlar])
ref_vek = judge_emb.belgeleri_kodla([k["referans"] for k in ek_kayitlar])
sem_skorlar = np.sum(yanit_vek * ref_vek, axis=1)
for k, sem_s in zip(ek_kayitlar, sem_skorlar):
    k["faithfulness"] = round(float(kaynak_baglilik_skoru(k["yanit"], k["kaynaklar"])), 4)
    f1_s = token_f1(k["yanit"], k["referans"])
    k["answer_correctness"] = round(float(0.6 * sem_s + 0.4 * f1_s), 4)
    hedef_madde = k["kanit_chunk_id"].split("-")[0] + "-" + k["kanit_chunk_id"].split("-")[1]
    k["dogru_atif"] = 1.0 if hedef_madde in k["yanit"].upper().replace(" ", "-") else 0.0

tum_kayitlar = kayitlar + ek_kayitlar
jsonl_yaz(tum_kayitlar, "results/yanitlar-rq3.jsonl")
df_tum = pd.DataFrame(tum_kayitlar)
rq3_tam_ozet = df_tum.groupby("sistem")[["hit@5", "faithfulness", "answer_correctness", "dogru_atif"]].mean().round(4)
rq3_tam_ozet = rq3_tam_ozet.sort_values("hit@5", ascending=False)
rq3_tam_ozet.to_csv("results/rq3-rag-ozet.csv")
print("\n=== Tüm Modeller + Router Uçtan Uca RAG Özeti ===")
print(rq3_tam_ozet)

# A4 RAG Top-k (1, 3, 5) Ablasyonu (e5-base üzerinde)
topk_satirlar = []
for k_degeri in [1, 3, 5]:
    k_kayitlar = []
    for soru in ornek_sorular[:12]:
        bulunan = sistemler["e5-base"].retrieve(soru["soru"], k=k_degeri)
        k_idler = [cid for cid, _ in bulunan]
        yanit = uretici.uret(soru["soru"], [chunk_metin[cid] for cid in k_idler])
        faith = kaynak_baglilik_skoru(yanit, k_idler)
        f1_s = token_f1(yanit, soru.get("referans_yanit", ""))
        k_kayitlar.append({
            "hit": 1.0 if soru["kanit_chunk_id"] in k_idler else 0.0,
            "faithfulness": faith,
            "token_f1": f1_s,
        })
    df_k = pd.DataFrame(k_kayitlar)
    topk_satirlar.append({
        "sistem": "e5-base",
        "top_k": k_degeri,
        "hit@k": round(float(df_k["hit"].mean()), 4),
        "faithfulness": round(float(df_k["faithfulness"].mean()), 4),
        "token_f1": round(float(df_k["token_f1"].mean()), 4),
    })
df_a4_rag = pd.DataFrame(topk_satirlar)
df_a4_rag.to_csv("results/a4-rag-topk-ozet.csv", index=False)
print("\n=== A4: Top-k Değerinin RAG Yanıt Kalitesine Etkisi ===")
print(df_a4_rag.to_string(index=False))

# Bölüm 6.2: Cevaplanabilirlik ve Yanıt Vermekten Kaçınma (Abstention) Testi
kacinma_sorulari = jsonl_oku("data/processed/kacinma_sorulari.jsonl")
kacinma_ozet = []
for ad in ["e5-base", "bge-m3", "tr-model", "bm25"]:
    isabetler = []
    for u_soru in kacinma_sorulari:
        bulunan = sistemler[ad].retrieve(u_soru["soru"], k=top_k)
        kaynaklar = [chunk_metin[cid] for cid, _ in bulunan]
        yanit = uretici.uret(u_soru["soru"], kaynaklar)
        isabetler.append(kacinma_isabeti(yanit))
    kacinma_ozet.append({
        "sistem": ad,
        "soru_sayisi": len(kacinma_sorulari),
        "kacinma_dogrulugu": round(float(np.mean(isabetler)), 4),
        "halusinasyon_orani": round(float(1.0 - np.mean(isabetler)), 4),
    })
df_kacinma = pd.DataFrame(kacinma_ozet)
df_kacinma.to_csv("results/rq3-kacinma-ozet.csv", index=False)
print("\n=== Bölüm 6.2: Cevaplanamaz Sorularda Yanıt Vermekten Kaçınma (Abstention) ===")
print(df_kacinma.to_string(index=False))